In [1]:
# ============================================================
# Task 5: Auto Tagging Support Tickets Using LLM
# ============================================================

# 1. Install dependencies
!pip install -q transformers pandas scikit-learn gradio

# 2. Imports
import pandas as pd
import numpy as np
from transformers import pipeline
import gradio as gr

# 3. Load a sample support ticket dataset (public)
# Using a known CSV from GitHub; if fails, fallback to built-in examples.
url = "https://raw.githubusercontent.com/martinsbalodis/support-tickets/master/support_tickets.csv"
try:
    df = pd.read_csv(url)
    tickets = df['issue_text'].tolist()[:20]  # first 20 tickets
    print(f"Loaded {len(tickets)} tickets from dataset.")
except Exception as e:
    print(f"Could not load from URL, using fallback examples. Error: {e}")
    tickets = [
        "My internet connection keeps dropping every hour.",
        "I was charged twice for the same subscription.",
        "How do I reset my password?",
        "The app crashes when I try to upload a photo.",
        "I want to cancel my account immediately.",
        "Your service is great, but I need a refund for last month.",
        "Can you change my billing address?",
        "The software does not install on Windows 11.",
        "I have not received my order confirmation email.",
        "Please upgrade my plan to premium.",
        "The website is down for the last 30 minutes.",
        "I can't log in to my account after the update."
    ]
    print(f"Using {len(tickets)} fallback examples.")

# 4. Define candidate tags (categories)
candidate_labels = [
    "billing",
    "technical issue",
    "account management",
    "cancellation",
    "product inquiry",
    "refund request"
]

# 5. Zero-shot classification pipeline (LLM)
print("Loading zero-shot classification model (facebook/bart-large-mnli)...")
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

def zero_shot_tag(ticket_text):
    """Return top-3 tags with scores using zero-shot classification."""
    result = classifier(ticket_text, candidate_labels)
    top3 = list(zip(result['labels'][:3], result['scores'][:3]))
    return top3

# 6. Few-shot approach (simulated using prompt engineering with the same model)
def few_shot_tag(ticket_text):
    """Append a few-shot prompt to the ticket text and classify."""
    prompt = f"""Examples:
- "I was charged twice" -> billing
- "Cannot log in" -> account management
- "Internet is down" -> technical issue
- "Cancel my plan" -> cancellation
- "Need product specs" -> product inquiry
- "Refund my payment" -> refund request

Now classify this ticket: {ticket_text}
Categories:"""
    combined = prompt + " " + ticket_text
    result = classifier(combined, candidate_labels)
    return list(zip(result['labels'][:3], result['scores'][:3]))

# 7. Compare zero-shot vs few-shot on first 5 tickets
print("\n===== Comparison: Zero-shot vs Few-shot =====")
for i, ticket in enumerate(tickets[:5]):
    print(f"\nTicket {i+1}: {ticket[:80]}...")
    zs = zero_shot_tag(ticket)
    fs = few_shot_tag(ticket)
    print(f"  Zero-shot: {zs}")
    print(f"  Few-shot:  {fs}")

# 8. Generate top-3 tags for all tickets (using zero-shot as final)
results = []
for ticket in tickets:
    tags = zero_shot_tag(ticket)
    results.append({
        "ticket": ticket,
        "top_tag_1": tags[0][0],
        "score_1": tags[0][1],
        "top_tag_2": tags[1][0],
        "score_2": tags[1][1],
        "top_tag_3": tags[2][0],
        "score_3": tags[2][1],
    })
df_results = pd.DataFrame(results)
print("\n===== First 5 results =====")
print(df_results.head())

# 9. Save results to CSV (optional)
df_results.to_csv("ticket_tags.csv", index=False)
print("\nResults saved to 'ticket_tags.csv'")

# 10. (Optional) Deploy a simple Gradio app for live tagging
def tag_ticket(ticket_text):
    tags = zero_shot_tag(ticket_text)
    return {tag: score for tag, score in tags}

iface = gr.Interface(
    fn=tag_ticket,
    inputs=gr.Textbox(lines=4, placeholder="Enter a support ticket...", label="Ticket Text"),
    outputs=gr.Label(num_top_classes=3, label="Top 3 Predicted Tags"),
    title="Support Ticket Auto-Tagging (LLM)",
    description="Zero-shot classification with facebook/bart-large-mnli. Predicts: billing, technical issue, account management, cancellation, product inquiry, refund request."
)
print("\nLaunching Gradio interface...")
iface.launch(share=True)

Could not load from URL, using fallback examples. Error: HTTP Error 404: Not Found
Using 12 fallback examples.
Loading zero-shot classification model (facebook/bart-large-mnli)...


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]


===== Comparison: Zero-shot vs Few-shot =====

Ticket 1: My internet connection keeps dropping every hour....
  Zero-shot: [('technical issue', 0.805084228515625), ('cancellation', 0.10691432654857635), ('product inquiry', 0.035820119082927704)]
  Few-shot:  [('technical issue', 0.3939521312713623), ('billing', 0.14366453886032104), ('account management', 0.14118032157421112)]

Ticket 2: I was charged twice for the same subscription....
  Zero-shot: [('billing', 0.8053515553474426), ('account management', 0.08052247762680054), ('refund request', 0.05198556184768677)]
  Few-shot:  [('billing', 0.37201157212257385), ('cancellation', 0.1622069925069809), ('account management', 0.15162436664104462)]

Ticket 3: How do I reset my password?...
  Zero-shot: [('account management', 0.4733678698539734), ('technical issue', 0.46419191360473633), ('product inquiry', 0.03522687032818794)]
  Few-shot:  [('billing', 0.27711835503578186), ('account management', 0.23292583227157593), ('technical issue